## Descarga de dependencias.

In [1]:
# 1. Desinstalar por completo los paquetes en conflicto
!pip uninstall -y transformers trl huggingface_hub accelerate peft bitsandbytes unsloth unsloth-zoo

# 2. Instalar la versión más reciente y compatible de Hugging Face y TRL
!pip install --no-cache-dir --upgrade huggingface_hub transformers trl accelerate peft bitsandbytes

# 3. Volver a instalar Unsloth desde su repositorio oficial
!pip install --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: huggingface_hub 1.11.0
Uninstalling huggingface_hub-1.11.0:
  Successfully uninstalled huggingface_hub-1.11.0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 12.4 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 157.4 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 307.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 142.1 MB/s eta 0:00:00a 0:00:0

## Prueba de modelo (Interferencia)


In [ ]:
import json
import torch
from pathlib import Path
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset
import ipywidgets as widgets
from IPython.display import display, FileLink, HTML

RUTA_BASE = Path("/kaggle/input/")

def buscar_datasets(ruta_raiz: Path) -> list:
    if not ruta_raiz.exists():
        return []
    return [{"nombre": p.name, "ruta": p} for p in sorted(ruta_raiz.rglob("*.json"))]

def cargar_json(ruta: Path) -> list:
    with ruta.open("r", encoding="utf-8-sig") as f:
        contenido = json.load(f)
    datos = contenido.get("root", []) if isinstance(contenido, dict) else contenido
    if not isinstance(datos, list) or not datos:
        raise ValueError(f"El dataset {ruta.name} está vacío o presenta una estructura no soportada.")
    return datos

class GestorGemma:
    def __init__(self):
        self.model = None
        self.tokenizer = None
        self.ultimo_modelo_exportado = None
        self.prompt_style = """A continuación hay una instrucción que describe una tarea. Escribe una respuesta que complete adecuadamente la solicitud.

### Instrucción:
{input}

### Respuesta:
{output}"""

    def entrenar(self, dataset_info: dict):
        print("🚀 Iniciando secuencia de carga para el modelo base Gemma 3 1B...")
        lista_datos = cargar_json(dataset_info["ruta"])
        max_seq_length = 2048
        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name="unsloth/gemma-3-1b-it", max_seq_length=max_seq_length,
            dtype=None, load_in_4bit=True,
        )
        self.model = FastLanguageModel.get_peft_model(
            self.model, r=16,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            lora_alpha=16, lora_dropout=0, bias="none",
            use_gradient_checkpointing="unsloth", random_state=3407,
        )
        textos = []
        for item in lista_datos:
            entrada = item.get("input") or item.get("palabras") or item.get("instruction") or ""
            salida = item.get("output") or item.get("oracion") or item.get("response") or ""
            if entrada and salida:
                textos.append(self.prompt_style.format(input=str(entrada), output=str(salida)) + self.tokenizer.eos_token)
        if not textos:
            raise ValueError("El JSON no contiene claves reconocibles.")
        dataset = Dataset.from_dict({"text": textos})
        pasos = max(20, len(dataset) * 2)
        trainer = SFTTrainer(
            model=self.model, tokenizer=self.tokenizer, train_dataset=dataset,
            dataset_text_field="text", max_seq_length=max_seq_length, packing=False,
            args=TrainingArguments(
                per_device_train_batch_size=2, gradient_accumulation_steps=4,
                warmup_steps=5, max_steps=pasos, learning_rate=2e-4,
                fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
                logging_steps=10, optim="adamw_8bit", weight_decay=0.01,
                lr_scheduler_type="linear", seed=3407, output_dir="outputs",
                report_to="none", disable_tqdm=True,
            ),
        )
        trainer.train()
        nombre_salida = f"Gemma3-{dataset_info['nombre'].replace('.json', '')}"
        self.ultimo_modelo_exportado = nombre_salida
        self.model.save_pretrained_gguf(nombre_salida, self.tokenizer, quantization_method="q4_k_m")
        print(f"🎉 Proceso finalizado: {nombre_salida}")

    def probar(self, input_text):
        if self.model is None or self.tokenizer is None:
            raise RuntimeError("Entrena el modelo antes de procesar muestras.")
        FastLanguageModel.for_inference(self.model)
        inputs = self.tokenizer([self.prompt_style.format(input=input_text, output="")], return_tensors="pt").to("cuda")
        outputs = self.model.generate(**inputs, max_new_tokens=64, use_cache=True)
        return self.tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

def procesar_entrada_json(raw_input: str) -> list:
    """Acepta un objeto, una lista JSON o una muestra JSON por línea."""
    raw_input = raw_input.strip()
    if not raw_input:
        return []
    try:
        datos = json.loads(raw_input)
        if isinstance(datos, dict):
            return [datos]
        if isinstance(datos, list):
            return [item for item in datos if isinstance(item, dict)]
    except json.JSONDecodeError:
        pass
    muestras = []
    for linea in raw_input.splitlines():
        linea = linea.strip()
        if not linea:
            continue
        try:
            muestra = json.loads(linea)
            if isinstance(muestra, dict):
                muestras.append(muestra)
        except json.JSONDecodeError:
            continue
    return muestras or [{"palabras": raw_input}]

def extraer_respuesta(raw_output: str, prompt_formateado: str) -> str:
    respuesta = raw_output.replace(prompt_formateado, "", 1).strip()
    if "### Respuesta:" in respuesta:
        respuesta = respuesta.split("### Respuesta:", 1)[1].strip()
    return respuesta

def construir_interfaz():
    gestor = GestorGemma()
    datasets_disponibles = buscar_datasets(RUTA_BASE)
    historial_evaluacion = []
    header = widgets.HTML("<h2>🎛️ Gemma 3 1B: entrenamiento y prueba JSON</h2>")
    if not datasets_disponibles:
        display(header, HTML("<b>⚠️ No se detectaron JSON en /kaggle/input/.</b>"))
        return
    opciones = {d["nombre"]: d for d in datasets_disponibles}
    dropdown = widgets.Dropdown(options=opciones.keys(), description="Dataset:")
    btn_entrenar = widgets.Button(description="1. Entrenar", button_style="primary")
    out_entrenar = widgets.Output()
    entrada = widgets.Textarea(
        description="Muestras:",
        placeholder='Una muestra por línea: {"palabras":"hungry pasta"}\nTambién acepta una lista JSON.',
        disabled=True, layout=widgets.Layout(width="95%", height="130px"),
    )
    btn_probar = widgets.Button(description="2. Procesar muestras", button_style="success", disabled=True)
    out_probar = widgets.Output()
    btn_descargar = widgets.Button(description="3. Enlaces GGUF", button_style="warning", disabled=True)
    out_descargar = widgets.Output()

    def al_entrenar(_):
        btn_entrenar.disabled = True
        with out_entrenar:
            out_entrenar.clear_output()
            try:
                gestor.entrenar(opciones[dropdown.value])
                print("✅ Entrenamiento terminado. Ya puedes introducir muestras.")
                btn_probar.disabled = False
                btn_descargar.disabled = False
                entrada.disabled = False
            except Exception as error:
                print(f"❌ Error: {error}")
            finally:
                btn_entrenar.disabled = False

    def al_probar(_):
        btn_probar.disabled = True
        with out_probar:
            out_probar.clear_output()
            texto = entrada.value.strip()
            if texto.lower() == "salir":
                print(json.dumps(historial_evaluacion, ensure_ascii=False, indent=2))
                entrada.value = ""
                btn_probar.disabled = False
                return
            muestras = procesar_entrada_json(texto)
            resultados = []
            for muestra in muestras:
                palabras = muestra.get("palabras") or muestra.get("input") or muestra.get("instruction")
                if not palabras:
                    continue
                try:
                    prompt = gestor.prompt_style.format(input=str(palabras), output="")
                    oracion = extraer_respuesta(gestor.probar(str(palabras)), prompt)
                    resultado = {"palabras": str(palabras), "oracion": oracion}
                    resultados.append(resultado)
                    historial_evaluacion.append(resultado)
                except Exception as error:
                    print(f"❌ Error procesando '{palabras}': {error}")
            print("\n".join(json.dumps(resultado, ensure_ascii=False) for resultado in resultados))
            entrada.value = ""
            btn_probar.disabled = False

    def al_descargar(_):
        with out_descargar:
            out_descargar.clear_output()
            archivos = sorted(Path.cwd().rglob("*.gguf"))
            if not archivos:
                print("⚠️ No se encontró ningún archivo .gguf.")
                return
            for archivo in archivos:
                display(FileLink(str(archivo), result_html_prefix="📄 "))

    btn_entrenar.on_click(al_entrenar)
    btn_probar.on_click(al_probar)
    btn_descargar.on_click(al_descargar)
    display(widgets.VBox([header, dropdown, btn_entrenar, out_entrenar, entrada, btn_probar, out_probar, btn_descargar, out_descargar]))

construir_interfaz()

In [19]:
import os
import shutil
from pathlib import Path
from IPython.display import display, FileLink, HTML

def descargar_modelo_especifico():
    # El sistema te pedirá que ingreses el nombre manualmente al ejecutar la celda
    dataset_name = input("👉 Por favor, inserta el nombre del dataset usado (ej. EnglishWords): ").strip()
    
    if not dataset_name:
        display(HTML("<b style='color:#f38ba8;'>⚠️ No ingresaste ningún nombre. Operación cancelada.</b>"))
        return

    nombre_objetivo = f"Gemma3-{dataset_name}"
    print(f"\n🔍 Buscando específicamente los artefactos para: '{nombre_objetivo}'...")
    
    # Buscamos cualquier archivo .gguf que contenga el nombre objetivo
    archivos_encontrados = list(Path(".").rglob(f"*{nombre_objetivo}*.gguf"))
    
    if not archivos_encontrados:
        display(HTML(f"""
        <div style='padding:15px; background-color:#311b22; border-left: 4px solid #f38ba8; color:#cdd6f4;'>
            <b>⚠️ No se encontró el archivo .gguf específico.</b><br>
            No hay rastros de un modelo llamado '{nombre_objetivo}'.
        </div>
        """))
        return

    display(HTML(f"<h3 style='color:#a6e3a1;'>📦 Modelo GGUF Listo para Descargar:</h3>"))
    
    for archivo in archivos_encontrados:
        # Extraemos el archivo de la subcarpeta que crea Unsloth
        ruta_raiz = Path(".") / archivo.name
        
        if archivo != ruta_raiz:
            try:
                shutil.move(str(archivo), str(ruta_raiz))
                archivo = ruta_raiz
            except Exception as e:
                pass # Si no se puede mover, usamos la ruta original
                
        # Mostramos la ruta y generamos el enlace
        tamaño_mb = archivo.stat().st_size / (1024 * 1024)
        display(HTML(f"<br><b>Archivo Confirmado:</b> <code>{archivo.name}</code> <i>({tamaño_mb:.2f} MB)</i>"))
        display(FileLink(str(archivo), result_html_prefix="⬇️ Haz clic aquí para descargar tu modelo: "))

# Ejecutamos la función
descargar_modelo_especifico()

👉 Por favor, inserta el nombre del dataset usado (ej. EnglishWords):  EnglishWords



🔍 Buscando específicamente los artefactos para: 'Gemma3-EnglishWords'...


In [ ]:
import json
import ipywidgets as widgets
from IPython.display import display, HTML

def iniciar_recolector():
    print("📝 Recolector de Datos para Corrección")
    print("Escribe tus entradas. Cuando termines, escribe 'salir' en cualquiera de los campos.")
    
    input_palabras = widgets.Text(description="Palabras:", placeholder="Ej: perro, gato, correr")
    input_oracion = widgets.Text(description="Oración:", placeholder="Ej: El perro corre por el parque.")
    btn_agregar = widgets.Button(description="➕ Agregar par", button_style="info")
    out_resultados = widgets.Output()
    
    registros = []
    
    def on_agregar(b):
        p = input_palabras.value.strip()
        o = input_oracion.value.strip()
        
        if p.lower() == "salir" or o.lower() == "salir":
            input_palabras.disabled = True
            input_oracion.disabled = True
            btn_agregar.disabled = True
            with out_resultados:
                out_resultados.clear_output()
                display(HTML("<h3 style='color:#a6e3a1;'>✨ Sesión finalizada. Copia el siguiente JSON y envíaselo al asistente para su corrección:</h3>"))
                json_output = json.dumps(registros, ensure_ascii=False, indent=4)
                print(json_output)
                display(widgets.HTML(f"<textarea style='width:100%; height:150px; background:#1e1e2e; color:#cdd6f4;'>{json_output}</textarea>"))
            return
            
        if p and o:
            registros.append({"palabras": p, "oracion": o})
            input_palabras.value = ""
            input_oracion.value = ""
            with out_resultados:
                out_resultados.clear_output()
                print(f"✅ Agregado con éxito. Total acumulado: {len(registros)}")
                print(json.dumps(registros, ensure_ascii=False, indent=2))
        else:
            with out_resultados:
                out_resultados.clear_output()
                print("⚠️ Por favor llena ambos campos o escribe 'salir' para terminar.")

    btn_agregar.on_click(on_agregar)
    
    display(widgets.VBox([
        widgets.HTML("<b>Introduce palabras/instrucción y la oración/respuesta esperada:</b>"),
        input_palabras,
        input_oracion,
        btn_agregar,
        out_resultados
    ]))

iniciar_recolector()